# Hindi → English: Causal wait-k simulation on Google FLEURS

**Inter-IIT Sarvam.ai ML track: entity accuracy vs latency.** This uses real public Hindi speech clip durations, real bilingual transcripts, and a pretrained Hindi→English MT model evaluated on each causal source prefix. **No dataset results are prefilled.** Run the notebook to plot entity recall versus simulated output lag.

**Pipeline:** Hindi speech metadata + Hindi transcript → prefix reveal → OPUS translation → irreversible wait-k commit → timed English speech events → entity accuracy against aligned English references.

**Limitations:** The Hindi transcript is an ASR oracle; source word ends are estimated unless Part C timings are added; ASR/MT/TTS timing budgets are simulated. FLEURS is read speech, not a code-switch conversation test. Set the Colab runtime to **T4 GPU** when possible.


## 1. Install packages (requires internet)

In [ ]:
%pip -q install "datasets[audio]>=3.2,<5" "transformers>=4.40,<5" "sentencepiece" "sacremoses" "spacy>=3.7,<4" "word2number" "pandas" "matplotlib" "soundfile"

In [ ]:
!python -m spacy download en_core_web_sm

## 2. Simulator and evaluator functions

English target word `i` (starting at 1) becomes eligible only after Hindi token `k+i` (or source EOS). Only observed source prefixes are ever translated. Previously emitted words are never edited.

In [ ]:
"""Causal, prefix-only wait-k Hindi->English translation simulation on Google FLEURS.

Uses oracle Hindi transcript and a proxy word timeline unless Part C CSV is provided.
Does NOT perform streaming ASR, actual TTS, or real-time inference scheduling.
"""
from __future__ import annotations
import argparse
import json
import math
import re
import string
import unicodedata
from collections import defaultdict
from functools import lru_cache
from pathlib import Path
from statistics import median
from time import perf_counter

import numpy as np
import pandas as pd

K_VALUES = (0, 1, 2, 3, 5, 8, 12, 999)
DATASET = "google/fleurs"
NAME_LABELS = {"PERSON", "ORG", "GPE", "LOC", "FAC", "PRODUCT", "EVENT", "NORP", "WORK_OF_ART"}
NUMBER_LABELS = {"CARDINAL", "ORDINAL", "MONEY", "PERCENT", "QUANTITY"}
DATE_LABELS = {"DATE", "TIME"}


def hindi_words(text: str) -> list[str]:
    """Tokenize Hindi by whitespace; retain mixed English words and digits."""
    return re.findall(r"\S+", str(text).strip())


def english_words(text: str) -> list[str]:
    """Split MT output into the target word units specified by the PS."""
    return re.findall(r"\S+", str(text).strip())


def proxy_end_times(words: list[str], duration_s: float) -> list[float]:
    """Approximate word end times from real clip duration using token character weights.

    These are NOT forced-aligned word timestamps and should not be called observed.
    """
    if not words or duration_s <= 0:
        raise ValueError("Need nonempty text and positive clip duration")
    weights = np.array([max(1.0, math.sqrt(max(1, len(w.strip(string.punctuation + '।'))))) for w in words])
    return (np.cumsum(weights) / weights.sum() * duration_s).tolist()


def read_word_timings(csv_path: str | None):
    """Read optional actual Part C word-end timings: id, word_index (0-based), end_s."""
    if not csv_path:
        return {}
    df = pd.read_csv(csv_path)
    needed = {"id", "word_index", "end_s"}
    if not needed.issubset(df.columns):
        raise ValueError(f"Part C timing CSV must have {sorted(needed)}")
    out = {}
    for sid, group in df.groupby("id"):
        group = group.sort_values("word_index")
        expected = list(range(len(group)))
        if group["word_index"].astype(int).tolist() != expected:
            raise ValueError(f"Missing or duplicate word indices in id={sid}")
        values = group["end_s"].astype(float).tolist()
        if any(v <= 0 for v in values) or any(x >= y for x, y in zip(values, values[1:])):
            raise ValueError(f"Invalid timestamps for id={sid}")
        out[str(sid)] = values
    return out


def word_duration_s(word: str, fixed: float = 0.30) -> float:
    """TTS duration proxy: a fixed number of seconds per spoken English word."""
    return fixed


def stream_waitk(words: list[str], ends_s: list[float], k: int, translate,
                 asr_delay_s: float = 0.12, mt_delay_s: float = 0.07,
                 frame_s: float = 0.08, tts_word_s: float = 0.30):
    """Causal wait-k event simulator: each emitted word uses ONLY the heard Hindi prefix.

    Standard wait-k target-word index i is 1-based: may emit only after min(n, k+i)
    Hindi tokens have arrived. At source EOS flush any remaining words of the final
    translation. Previously emitted speech is irrevocable.
    """
    if k < 0 or len(words) != len(ends_s) or not words:
        raise ValueError("k >= 0 and one valid end time per source word required")
    if any(x >= y for x, y in zip(ends_s, ends_s[1:])):
        raise ValueError("Source word end times must strictly increase")
    emitted = []
    events = []
    tts_available = 0.0
    n = len(words)
    for r in range(1, n + 1):
        # Translate only an actually observed Hindi prefix, with no future context.
        hyp = english_words(translate(" ".join(words[:r])))
        # At source EOS, remaining output is flushed even when k > source length.
        max_target_words = len(hyp) if r == n else max(0, r - k)
        while len(emitted) < min(max_target_words, len(hyp)):
            i = len(emitted)
            token = hyp[i]
            ready_s = math.ceil((ends_s[r - 1] + asr_delay_s + mt_delay_s) / frame_s) * frame_s
            start_s = max(ready_s, tts_available)
            end_s = start_s + word_duration_s(token, tts_word_s)
            tts_available = end_s
            emitted.append(token)
            events.append({"target_index": i + 1, "spoken_word": token,
                           "heard_source_words": r, "source_heard_s": float(ends_s[r - 1]),
                           "speech_start_s": start_s, "speech_end_s": end_s})
    return " ".join(emitted), events


def entity_kind(ent_label: str) -> str | None:
    """Coarsen spaCy NER tags into named, numeric, and temporal classes."""
    if ent_label in NAME_LABELS:
        return "name"
    if ent_label in NUMBER_LABELS:
        return "number"
    if ent_label in DATE_LABELS:
        return "date"
    return None


def normalized_string(s: str) -> str:
    """Normalize punctuation and case for strict English surface-form matching."""
    t = unicodedata.normalize("NFKC", str(s)).casefold()
    t = re.sub(r"[^\w\s]", " ", t, flags=re.UNICODE)
    return " ".join(t.split())


def entity_key(ent_text: str, kind: str) -> str:
    """Convert common English number expressions to a normalized numeric value."""
    norm = normalized_string(ent_text)
    if kind == "number":
        try:
            from word2number import w2n
            return f"numeric:{w2n.word_to_num(norm.replace(',', ''))}"
        except (ValueError, IndexError, TypeError, AttributeError):
            pass
        digits = re.sub(r"[^0-9.\-]", "", ent_text)
        if re.fullmatch(r"-?\d+(?:\.\d+)?", digits):
            return f"numeric:{float(digits):g}"
    return norm


def entities(text: str, nlp) -> list[dict]:
    """Extract reference or predicted English entities using the same NER model."""
    doc = nlp(text)
    found = {}
    for e in doc.ents:
        kind = entity_kind(e.label_)
        if not kind:
            continue
        key = entity_key(e.text, kind)
        if key:
            found[(kind, key)] = {"kind": kind, "surface": e.text, "key": key}
    # Digit sequences are still tested if the NER model fails to tag them.
    for m in re.finditer(r"(?<![\w])\d[\d,]*(?:\.\d+)?(?![\w])", text):
        key = entity_key(m.group(), "number")
        if key and ("number", key) not in found:
            found[("number", key)] = {"kind": "number", "surface": m.group(), "key": key}
    return list(found.values())


def entity_scores(ref_ents, pred_ents, pred_text=""):
    """Count entities recovered either by prediction NER or exact reference surface match.

    Reference-surface fallback avoids penalizing correct named entities if the NER
    tagger misses them in the (possibly uncapitalized) prediction text.
    """
    ref_set = {(x["kind"], x["key"]) for x in ref_ents}
    pred_set = {(x["kind"], x["key"]) for x in pred_ents}
    norm_pred = f" {normalized_string(pred_text)} "
    hits = 0
    for ent in ref_ents:
        if (ent["kind"], ent["key"]) in pred_set:
            hits += 1
        elif f" {normalized_string(ent['surface'])} " in norm_pred:
            hits += 1
    return hits, len(ref_set), max(len(pred_set), hits)


def latency_stats(events: list[dict], duration_s: float, n_src: int):
    """Compute token Average Lagging and time-based output-start lag proxy."""
    if not events:
        return {"token_AL": float("nan"), "lag_s": float("nan"),
                "first_audio_s": float("nan"), "tail_latency_s": float("nan")}
    m = len(events)
    n = n_src
    tau = next((idx for idx, e in enumerate(events, 1) if e["heard_source_words"] == n), m)
    token_al = float(np.mean([e["heard_source_words"] - (j - 1) * n / m
                              for j, e in enumerate(events[:tau], 1)]))
    # Compare output start to an ideal, evenly paced target speech track.
    lags = [max(0, e["speech_start_s"] - (j - 1) * duration_s / m)
            for j, e in enumerate(events, 1)]
    return {"token_AL": token_al, "lag_s": float(np.mean(lags)),
            "first_audio_s": float(events[0]["speech_start_s"]),
            "tail_latency_s": float(events[-1]["speech_end_s"] - duration_s)}


def load_fleurs_pairs(max_clips: int, require_entity: bool, nlp):
    """Stream parallel FLEURS test sentences, join by shared ID, use Hindi audio length."""
    from datasets import load_dataset
    print("Streaming English reference text (en_us, test) ...", flush=True)
    eng_ds = load_dataset(DATASET, "en_us", split="test", streaming=True).remove_columns("audio")
    english = {str(x["id"]): x.get("raw_transcription") or x["transcription"] for x in eng_ds}
    print(f"Loaded {len(english)} English references. Streaming Hindi clips ...", flush=True)
    hindi_ds = load_dataset(DATASET, "hi_in", split="test", streaming=True).remove_columns("audio")
    count_scanned = 0
    selected = []
    for row in hindi_ds:
        count_scanned += 1
        sid = str(row["id"])
        if sid not in english:
            continue
        hi = row.get("raw_transcription") or row["transcription"]
        en = english[sid]
        words = hindi_words(hi)
        # FLEURS has 16 kHz audio; prefer metadata sample count to avoid reading audio.
        duration = float(row["num_samples"]) / 16000.0
        if not (4 <= len(words) <= 55 and 2.0 <= duration <= 36.0):
            continue
        refs = entities(en, nlp)
        if require_entity and not refs:
            continue
        selected.append({"id": sid, "hindi": hi, "reference_english": en,
                         "duration_s": duration, "words": words, "gold_entities": refs})
        if len(selected) >= max_clips:
            break
    print(f"Selected {len(selected)} clips from {count_scanned} Hindi rows."
          " Entity-bearing selection is NOT a random benchmark sample.", flush=True)
    if not selected:
        raise RuntimeError("No paired FLEURS examples found. Verify split, FLEURS IDs, and network access.")
    return selected


def load_translator(model_id="Helsinki-NLP/opus-mt-hi-en", device="auto"):
    """Load one pretrained Hindi->English model and cache translations of source prefixes."""
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    use_device = "cuda" if device == "auto" and torch.cuda.is_available() else ("cpu" if device == "auto" else device)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(use_device).eval()
    run_ms = []

    @lru_cache(maxsize=10000)
    def translate(txt: str) -> str:
        inputs = tokenizer(txt, return_tensors="pt", truncation=True, max_length=256)
        inputs = {key: value.to(use_device) for key, value in inputs.items()}
        if use_device.startswith("cuda"):
            torch.cuda.synchronize()
        tic = perf_counter()
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=90, num_beams=1)
        if use_device.startswith("cuda"):
            torch.cuda.synchronize()
        run_ms.append((perf_counter() - tic) * 1000)
        return tokenizer.decode(output[0], skip_special_tokens=True).strip()

    return translate, run_ms, use_device


def plot_tradeoff(summary: pd.DataFrame, output_path: Path):
    """Save observed entity recall vs ASSUMED-streaming timing proxy chart."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    df = summary.sort_values("mean_lag_s")
    fig, ax = plt.subplots(figsize=(9.3, 6.2))
    ax.plot(df["mean_lag_s"], df["entity_recall_pct"], "-o", lw=2, ms=8)
    for _, row in df.iterrows():
        ax.annotate(row["k_label"], (row["mean_lag_s"], row["entity_recall_pct"]),
                    xytext=(6, 7), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Mean output-start lag vs ideal speech clock (seconds; proxy)")
    ax.set_ylabel("Micro entity recall (%)  |  reference NER")
    ax.set_title("FLEURS Hindi → English: Causal wait-k tradeoff\n"
                 "Actual model translations; assumed ASR/MT/TTS timing")
    ax.set_ylim(-5, 105)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=185)
    plt.close(fig)


def evaluate(clips, translate, nlp, k_values, output_dir: Path, supplied_timings=None,
             tts_word_s=0.30, asr_delay_s=0.12, mt_delay_s=0.07):
    """Run all k settings over the same clips; save per-utterance, schedule and plot."""
    supplied_timings = supplied_timings or {}
    output_dir.mkdir(parents=True, exist_ok=True)
    rows, schedules = [], []
    gold_count = sum(len(c["gold_entities"]) for c in clips)
    print(f"Reference English NER found {gold_count} distinct entities across {len(clips)} clips.")
    if not gold_count:
        raise ValueError("No reference entities: increase sample count or change filter")
    for ci, clip in enumerate(clips):
        words = clip["words"]
        ends_s = supplied_timings.get(str(clip["id"]))
        used_aligned_timing = ends_s is not None
        if ends_s is None:
            ends_s = proxy_end_times(words, clip["duration_s"])
        if len(ends_s) != len(words):
            raise ValueError(f"clip id {clip['id']}: {len(words)} words but {len(ends_s)} timestamps")
        for k in k_values:
            pred, events = stream_waitk(words, ends_s, k, translate,
                                        asr_delay_s=asr_delay_s, mt_delay_s=mt_delay_s,
                                        tts_word_s=tts_word_s)
            hit, gold, guessed = entity_scores(clip["gold_entities"], entities(pred, nlp), pred)
            latency = latency_stats(events, clip["duration_s"], len(words))
            rows.append({"id": clip["id"], "k": k, "k_label": "full" if k >= len(words) else str(k),
                         "duration_s": clip["duration_s"], "n_hindi_words": len(words),
                         "n_english_words": len(events), "entity_hits": hit, "reference_entities": gold,
                         "predicted_entities": guessed, "hindi": clip["hindi"],
                         "reference_english": clip["reference_english"], "streamed_english": pred,
                         "ref_entity_list": json.dumps(clip["gold_entities"], ensure_ascii=False),
                         "word_times": "aligned_part_c" if used_aligned_timing else "duration_weighted_proxy",
                         **latency})
            if ci == 0:
                for ev in events:
                    schedules.append({"id": clip["id"], "k": k, **ev})
        if (ci + 1) % 5 == 0 or ci == len(clips) - 1:
            print(f"Simulated {ci+1}/{len(clips)} clips", flush=True)
    details = pd.DataFrame(rows)
    summary = details.groupby("k", sort=True).agg(
        entity_hits=("entity_hits", "sum"), reference_entities=("reference_entities", "sum"),
        predicted_entities=("predicted_entities", "sum"), clips=("id", "nunique"),
        mean_lag_s=("lag_s", "mean"), mean_token_AL=("token_AL", "mean"),
        mean_first_audio_s=("first_audio_s", "mean"), mean_tail_latency_s=("tail_latency_s", "mean"))
    summary["entity_recall_pct"] = summary["entity_hits"] / summary["reference_entities"] * 100
    summary["entity_precision_pct"] = (summary["entity_hits"] / summary["predicted_entities"].replace(0, np.nan) * 100).fillna(0)
    summary = summary.reset_index()
    summary["k_label"] = summary["k"].map(lambda k: "full" if k >= max(len(c["words"]) for c in clips) else str(k))
    details.to_csv(output_dir / "per_clip_results.csv", index=False)
    summary.to_csv(output_dir / "waitk_summary.csv", index=False)
    pd.DataFrame(schedules).to_csv(output_dir / "first_clip_event_schedule.csv", index=False)
    plot_tradeoff(summary, output_dir / "entity_vs_latency.png")
    return summary, details


def self_test():
    """Use a handwritten mock to assert causal behavior; NOT benchmark performance."""
    src = "मैं आज दिल्ली नहीं मुंबई जा रहा हूँ".split()
    times = [float(i + 1) * 0.45 for i in range(len(src))]
    def mock(prefix):
        return "Mumbai is the destination today" if "मुंबई" in prefix.split() else "Delhi is the destination today"
    bad, ev_bad = stream_waitk(src, times, k=0, translate=mock)
    good, ev_good = stream_waitk(src, times, k=5, translate=mock)
    assert bad.split()[0] == "Delhi", bad
    assert good.split()[0] == "Mumbai", good
    assert all(ev_bad[i]["speech_end_s"] <= ev_bad[i+1]["speech_start_s"] + 1e-9 for i in range(len(ev_bad)-1))
    assert all(ev_good[i]["speech_end_s"] <= ev_good[i+1]["speech_start_s"] + 1e-9 for i in range(len(ev_good)-1))
    assert ev_good[0]["heard_source_words"] >= 6
    assert "Mumbai" not in bad.split()[:1]
    print("PASS causal wait-k smoke test:\n  k=0:", bad, "\n  k=5:", good)
    return src, times, bad, good


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--n", type=int, default=20, help="Max entity-bearing Hindi-English FLEURS pairs")
    p.add_argument("--k", nargs="+", type=int, default=list(K_VALUES))
    p.add_argument("--output-dir", default="results")
    p.add_argument("--part-c-timings", default=None, help="Optional id,word_index,end_s CSV with real word ends")
    p.add_argument("--model", default="Helsinki-NLP/opus-mt-hi-en")
    p.add_argument("--device", default="auto")
    p.add_argument("--tts-word-s", type=float, default=0.30)
    p.add_argument("--asr-delay-s", type=float, default=0.12)
    p.add_argument("--mt-delay-s", type=float, default=0.07)
    p.add_argument("--include-entity-free", action="store_true")
    p.add_argument("--self-test", action="store_true")
    a = p.parse_args()
    if a.self_test:
        self_test()
        return
    if a.n < 1 or not a.k or min(a.k) < 0:
        p.error("--n must be positive, and --k must contain nonnegative integers")
    import spacy
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError as e:
        raise RuntimeError("Run: python -m spacy download en_core_web_sm") from e
    clips = load_fleurs_pairs(a.n, require_entity=not a.include_entity_free, nlp=nlp)
    translate, mt_times_ms, device = load_translator(a.model, a.device)
    out = Path(a.output_dir)
    aligned = read_word_timings(a.part_c_timings)
    summary, details = evaluate(clips, translate, nlp, sorted(set(a.k)), out, aligned,
                                a.tts_word_s, a.asr_delay_s, a.mt_delay_s)
    stats = {"dataset": DATASET, "dataset_split": "test", "language": "hi_in -> en_us",
             "translation_model": a.model, "device": device, "number_clips": len(clips),
             "k_values": sorted(set(a.k)), "selected_entity_bearing": not a.include_entity_free,
             "word_alignment": "actual Part C when provided; otherwise duration-weighted proxy",
             "pipeline": "oracle Hindi transcript -> prefix-only MT -> wait-k commit -> fixed duration TTS schedule",
             "mt_delay_budget_s": a.mt_delay_s,
             "measured_offline_mt_inference_median_ms": median(mt_times_ms) if mt_times_ms else None,
             "measured_offline_mt_inference_calls": len(mt_times_ms),
             "warning": "ASR and TTS are simulated, not executed; timing budgets are assumed. Word timestamps default to proxy."
             }
    (out / "experiment_metadata.json").write_text(json.dumps(stats, indent=2, ensure_ascii=False))
    print(summary.round(3).to_string(index=False))
    # Smallest output lag within two percentage points of best observed entity recall.
    max_recall = summary["entity_recall_pct"].max()
    candidates = summary[summary["entity_recall_pct"] >= max_recall - 2.0]
    choice = candidates.sort_values(["mean_lag_s", "k"]).iloc[0]
    print(f"\nLatency-first candidate (within 2 pp of highest entity recall): "
          f"k={int(choice['k'])} | recall={choice['entity_recall_pct']:.1f}% "
          f"| lag={choice['mean_lag_s']:.2f}s")
    print("Treat this as a small-sample candidate, NOT an unconditional production setting.")
    print(f"\nResults and chart: {out.resolve()}")



## 3. Causal gate unit test (handwritten mock, **not** dataset scores)

In [ ]:
src, ends, k0_output, k5_output = self_test()
print('A later Hindi correction can change the first English word only if we waited.')

## 4. Download real FLEURS Hindi examples and matching English references

Test set: `google/fleurs`, `hi_in` and `en_us`. The same FLEURS sentence ID pairs Hindi audio duration/transcript with English reference text. Entity-bearing clips are selected deliberately. Hindi word-end estimates use duration and approximate relative token lengths; replace with Part C alignments as soon as you have them.

In [ ]:
NUM_CLIPS = 16                          # set to 8 for a faster run
K_VALUES = [0, 1, 2, 3, 5, 8, 12, 999]    # k=999 is effectively full context
PART_C_CSV = None                       # set to path of id,word_index,end_s CSV, if available
OUTPUT_DIR = Path('/content/waitk_results') if Path('/content').exists() else Path('waitk_results')

import spacy
nlp = spacy.load('en_core_web_sm')
clips = load_fleurs_pairs(NUM_CLIPS, require_entity=True, nlp=nlp)
for clip in clips[:3]:
    print(f"ID={clip['id']} | duration={clip['duration_s']:.2f}s")
    print('HI:', clip['hindi'])
    print('EN:', clip['reference_english'])
    print('Reference entities:', clip['gold_entities'], '\n')
print('Check the Hindi-English pairings manually before interpreting metrics.')

## 5. Run pretrained Hindi → English translation **on each observed prefix**

Model: `Helsinki-NLP/opus-mt-hi-en`. This stage performs real MT inference, cached across `k` values. The scheduler separately assumes fixed 120ms streaming ASR + 70ms translation + 80ms frame quantization, plus 300ms English word duration; these are **not measured server timings**.

In [ ]:
translate, measured_mt_ms, device = load_translator('Helsinki-NLP/opus-mt-hi-en')
print('Device:', device)
part_c_times = read_word_timings(PART_C_CSV)
summary, details = evaluate(clips, translate, nlp, K_VALUES, OUTPUT_DIR,
                            supplied_timings=part_c_times,
                            tts_word_s=0.30, asr_delay_s=0.12, mt_delay_s=0.07)
from IPython.display import display
display(summary[['k_label','entity_recall_pct','entity_precision_pct',
                 'mean_lag_s','mean_first_audio_s','mean_tail_latency_s','mean_token_AL']].round(3))

## 6. Entity accuracy vs latency (actual model outputs; simulated timeline)

Vertical axis: English reference entity recall from the committed translation. Horizontal axis: assumed English output-start lag versus an ideal synchronized speech clock. The plot is **generated from real model inference**, not example data.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT_DIR / 'entity_vs_latency.png')))
print('Unique prefix translations:', len(measured_mt_ms))
print('Median offline model inference (ms):', round(median(measured_mt_ms), 1) if measured_mt_ms else None)

## 6b. Pick a latency-first candidate k near the observed accuracy plateau

In [ ]:
best_recall = summary['entity_recall_pct'].max()
near_best = summary[summary['entity_recall_pct'] >= best_recall - 2.0]
best = near_best.sort_values(['mean_lag_s','k']).iloc[0]
print(f"Candidate: k={int(best['k'])}, recall={best['entity_recall_pct']:.1f}%, output lag={best['mean_lag_s']:.2f}s")
print('Selection rule: minimum lag within 2 percentage points of best recall. Small-sample only.')

## 7. Inspect how translations and errors change with k

In [ ]:
for cid in details['id'].unique()[:3]:
    ex = details[details.id == cid].sort_values('k')
    print('\n===== CLIP ID', cid, '=====')
    print('HI:', ex.iloc[0]['hindi'])
    print('Reference EN:', ex.iloc[0]['reference_english'])
    for r in ex.itertuples():
        print(f"k={r.k:>3}: entities={r.entity_hits}/{r.reference_entities} | {r.streamed_english}")
print('\nFirst clip English speech event schedule:')
display(pd.read_csv(OUTPUT_DIR / 'first_clip_event_schedule.csv').head(20))

## 8. Save and download the complete experiment results

In [ ]:
import shutil
metadata = {
 'dataset':'google/fleurs', 'configs':['hi_in','en_us'], 'split':'test',
 'n_entity_bearing_clips':len(clips), 'k_values':K_VALUES,
 'model':'Helsinki-NLP/opus-mt-hi-en', 'device':device,
 'measured_prefix_inference_calls':len(measured_mt_ms),
 'median_measured_mt_ms':float(median(measured_mt_ms)) if measured_mt_ms else None,
 'word_timings':'Part C' if PART_C_CSV else 'duration-weighted proxy',
 'asr':'oracle transcript', 'tts':'300ms fixed word duration',
 'latency':'assumed ASR=120ms, MT=70ms, scheduling=80ms frames; not measured'}
(OUTPUT_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2))
archive = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=str(OUTPUT_DIR))
print('Saved:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass

## Recommendation / next experiment

Look for the **smallest `k` near the entity accuracy plateau**, and identify individual errors due to delayed Hindi verbs, named entities and amounts. Do not present these assumed timings as production S2ST latency. Repeat with real Part C forced-aligned word times, actual streaming ASR outputs, measured MT and TTS decoding, and a manually scored code-switch/correction set.

Dataset: [Google FLEURS](https://huggingface.co/datasets/google/fleurs) (CC BY 4.0). Model: [Helsinki-NLP OPUS MT hi-en](https://huggingface.co/Helsinki-NLP/opus-mt-hi-en) (Apache 2.0).